# Paper figures — Figure 3 and Figure 4

**Figure reconstruction from saved artifacts.**

- **Figure 3**: dose–response on instruct and base models: % change in
  |ΔS| relative to baseline (α = 1), one curve per model, built from the
  8 `dose_response_<model>_<variant>.pkl` pickles written by the
  `dose_response/` notebooks.
- **Figure 4**: association–differentiation gap on instruct models:
  |ΔS| vs |ΔK| grouped bars with the K/S ratio annotated, built from the
  4 instruct `results_<model>_instruct_stage4_ko.pkl` pickles written by the
  `binding_knockout/` notebook.

No hardcoded values: every number is read from the artifacts found
recursively under `./results/`. Figures are saved to `./results/figures/`.

In [ ]:
import os
# Locate the repo root (the directory containing common/), whatever the kernel cwd
_p = os.path.abspath(".")
REPO_ROOT = _p if os.path.isdir(os.path.join(_p, "common")) else os.path.abspath("..")
assert os.path.isdir(os.path.join(REPO_ROOT, "common")), (
    "Cannot locate the repo root: run this notebook from its own directory or the repo root")
import sys
sys.path.insert(0, REPO_ROOT)
import glob
import pickle

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

RESULTS_ROOT = os.path.join(REPO_ROOT, "results") + os.sep
FIG_DIR = Path(REPO_ROOT) / "results" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)


def find_artifact(fname):
    """Recursively locate a result pickle under RESULTS_ROOT."""
    matches = glob.glob(os.path.join(RESULTS_ROOT, "**", fname), recursive=True)
    if not matches:
        raise FileNotFoundError(f"{fname} not found under {RESULTS_ROOT}")
    if len(matches) > 1:
        print(f"  WARNING: {len(matches)} matches for {fname}, using {matches[0]}")
    return matches[0]


# Paper display order and historical key asymmetry: the base pipeline uses
# "gemma" where the instruct pipeline uses "gemma2".
MODELS = [
    # (display name, instruct key, base key)
    ("Mistral", "mistral", "mistral"),
    ("Gemma",   "gemma2",  "gemma"),
    ("Llama",   "llama",   "llama"),
    ("Nemo",    "nemo",    "nemo"),
]

# Fixed categorical assignment (colorblind-safe Okabe-Ito subset): the same
# model keeps the same color and marker in every panel and every figure.
MODEL_COLORS  = {"Mistral": "#0072B2", "Gemma": "#E69F00",
                 "Llama": "#009E73",  "Nemo": "#D55E00"}
MODEL_MARKERS = {"Mistral": "o", "Gemma": "s", "Llama": "^", "Nemo": "D"}

## Figure 3 — dose–response (instruct + base, 4 models)

In [ ]:
# ================================================================
# FIGURE 3 — dose-response: % change in |ΔS| relative to baseline (α=1)
# ================================================================
dose = {}   # (display name, variant) -> (alphas, % change in |ΔS|)
for disp, key_i, key_b in MODELS:
    for variant, key in (("instruct", key_i), ("base", key_b)):
        p = find_artifact(f"dose_response_{key}_{variant}.pkl")
        with open(p, "rb") as f:
            d = pickle.load(f)
        results_dr = d["results_dr"]
        alphas = sorted(results_dr.keys())
        assert 1.0 in results_dr, f"{p}: no alpha=1 point"
        ref = abs(results_dr[1.0]["delta"])
        assert ref > 1e-12, f"{p}: |ΔS| at alpha=1 is ~0"
        pct = [(abs(results_dr[a]["delta"]) - ref) / ref * 100.0 for a in alphas]
        dose[(disp, variant)] = (alphas, pct)
        print(f"  {disp:8s} {variant:8s}  |ΔS|(α=1) = {ref:.4f}   "
              f"[{os.path.relpath(p, RESULTS_ROOT)}]")

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for ax, variant, title in zip(axes, ("instruct", "base"), ("Instruct", "Base")):
    for disp, _ki, _kb in MODELS:
        alphas, pct = dose[(disp, variant)]
        ax.plot(alphas, pct, linestyle="-", linewidth=2,
                marker=MODEL_MARKERS[disp], markersize=5,
                color=MODEL_COLORS[disp], label=disp)
    ax.axhline(0, color="gray", linewidth=0.6, linestyle="--")
    ax.axvline(1.0, color="gray", linewidth=0.6, linestyle=":")
    ax.set_xlabel(r"$\alpha$")
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
axes[0].set_ylabel(r"% change in $|\Delta S|$")
axes[0].legend(fontsize=9, frameon=False)
plt.tight_layout()
fig3_path = FIG_DIR / "fig3_dose_response.png"
fig.savefig(fig3_path, dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved {fig3_path}")

## Figure 4 — association–differentiation gap (instruct)

In [ ]:
# ================================================================
# FIGURE 4 — association-differentiation gap (instruct models)
# |ΔS| and |ΔK| are the absolute means of the per-pair baseline diffs
# stored in the stage4_ko pickles (same definition as Table 4).
# ================================================================
rows = []
for disp, key_i, _kb in MODELS:
    p = find_artifact(f"results_{key_i}_instruct_stage4_ko.pkl")
    with open(p, "rb") as f:
        store = pickle.load(f)
    abs_dS = abs(float(np.mean(store["knockout"]["binding"]["diffs_base"])))
    abs_dK = abs(float(np.mean(store["knockout"]["knowledge"]["diffs_base"])))
    rows.append({"model": disp, "abs_dS": abs_dS, "abs_dK": abs_dK,
                 "ratio": abs_dK / abs_dS})
    print(f"  {disp:8s} |ΔS| = {abs_dS:6.2f}   |ΔK| = {abs_dK:6.2f}   "
          f"K/S = {abs_dK / abs_dS:.2f}   [{os.path.relpath(p, RESULTS_ROOT)}]")

xs = np.arange(len(rows))
w = 0.38
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(xs - w / 2, [r["abs_dS"] for r in rows], w,
       color="#0072B2", label=r"$|\Delta S|$ (binding)")
ax.bar(xs + w / 2, [r["abs_dK"] for r in rows], w,
       color="#E69F00", label=r"$|\Delta K|$ (knowledge)")
for x, r in zip(xs, rows):
    ax.text(x + w / 2, r["abs_dK"] * 1.01, f"{r['ratio']:.2f}x",
            ha="center", va="bottom", fontsize=9, color="black")
ax.set_xticks(xs)
ax.set_xticklabels([r["model"] for r in rows])
ax.set_ylabel("pairing effect (absolute)")
ax.set_title("Association-differentiation gap (instruct models); "
             "annotation = K/S ratio")
ax.legend(frameon=False)
ax.grid(True, axis="y", alpha=0.3)
ax.set_axisbelow(True)
plt.tight_layout()
fig4_path = FIG_DIR / "fig4_association_differentiation_gap.png"
fig.savefig(fig4_path, dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved {fig4_path}")